# Búsqueda informada

**Facundo A. Lucianna - Inteligencia Artificial - CEIA - FIUBA**

Los algoritmos de la notebook anterior no saben nada del problema: para ellos, todos los estados de la frontera son igual de prometedores. Los algoritmos de **búsqueda informada** usan una **función heurística** `h(n)`, que estima cuánto falta para llegar desde el estado del nodo `n` hasta el objetivo.

En esta notebook vamos a implementar los dos algoritmos de búsqueda informada que vimos en la teoría:

1. Búsqueda voraz (*greedy*) primero el mejor
2. Búsqueda A*

Los dos usan una cola prioritaria, igual que la búsqueda de costo uniforme. Lo único que cambia es la prioridad con la que se ordenan los nodos.

In [1]:
from aima_libs.aima import PriorityQueue
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi


def create_problem(number_disks: int = 5) -> ProblemHanoi:
    """Crea el problema de la Torre de Hanoi con todos los discos en la primera
    varilla, y el objetivo de llevarlos a la tercera."""
    list_disks = list(range(number_disks, 0, -1))
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    return ProblemHanoi(initial=initial_state, goal=goal_state)

---

## La función heurística

Vamos a usar la heurística que se explica en el documento [Funciones heurísticas - Torre de Hanoi](../heuristic_functions.md): **penaliza con un punto cada disco que no está en su posición correcta**. Un disco está en su posición correcta si está en la tercera varilla y debajo de él están todos los discos más grandes, en orden. Es decir, contamos cuántos discos ya están bien apilados desde la base de la tercera varilla, y se los restamos al total:

In [2]:
def misplaced_disks(state: StatesHanoi) -> int:
    """
    Heurística para la Torre de Hanoi: cantidad de discos que no están en su
    posición correcta.

    Args:
        state (StatesHanoi): Estado a evaluar.

    Returns:
        int: Cantidad de discos mal ubicados.
    """
    goal_rod = state.rods[2]
    correct = 0
    # Recorremos la tercera varilla desde la base, comparando con el orden objetivo
    for expected_disk, disk in zip(range(state.number_of_disks, 0, -1), goal_rod):
        if disk != expected_disk:
            break
        correct += 1
    return state.number_of_disks - correct

Probemos con los tres estados del documento, con 5 discos:

In [3]:
three_correct = StatesHanoi([2, 1], [], [5, 4, 3], max_disks=5)
one_correct = StatesHanoi([4, 3, 2, 1], [], [5], max_disks=5)
initial = StatesHanoi([5, 4, 3, 2, 1], [], [], max_disks=5)

for state in [three_correct, one_correct, initial]:
    print(f"{state}  ->  h = {misplaced_disks(state)}")

HanoiState: 2 1 |  | 5 4 3  ->  h = 2
HanoiState: 4 3 2 1 |  | 5  ->  h = 4
HanoiState: 5 4 3 2 1 |  |   ->  h = 5


---

## Búsqueda voraz (*greedy*) primero el mejor

La búsqueda voraz expande siempre el nodo que **parece estar más cerca del objetivo**, es decir, el de menor `h(n)`. No le importa cuánto costó llegar hasta ese nodo, solo cuánto cree que falta.

El código es el mismo que el de costo uniforme de la notebook anterior. Lo único que cambia es la función de prioridad de la cola, que pasa a ser `h(n)`. Por eso armamos una función general, que recibe la función de prioridad como argumento:

In [4]:
def best_first_search(number_disks: int, priority) -> tuple:
    """
    Búsqueda primero el mejor: expande siempre el nodo de la frontera con menor
    valor de `priority`.

    Args:
        number_disks (int): Cantidad de discos.
        priority (callable): Función que recibe un nodo y devuelve su prioridad.

    Returns:
        tuple: (último nodo de la solución, diccionario con métricas)
    """
    problem = create_problem(number_disks)

    frontier = PriorityQueue(order="min", f=priority)
    frontier.append(NodeHanoi(problem.initial))
    explored = set()
    node_explored = 0

    while len(frontier) != 0:
        _, node = frontier.pop()

        # Si ya lo exploramos antes, lo salteamos
        if node.state in explored:
            continue

        node_explored += 1
        explored.add(node.state)

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": node.depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics

        for next_node in node.expand(problem):
            if next_node.state not in explored:
                frontier.append(next_node)

    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": 0,
        "max_depth": None,
        "cost_total": None,
    }
    return None, metrics


def greedy_search(number_disks: int = 5) -> tuple:
    """Búsqueda voraz: la prioridad es h(n)."""
    return best_first_search(number_disks, lambda node: misplaced_disks(node.state))

In [5]:
solution, metrics = greedy_search(number_disks=5)

for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 107
states_visited: 107
nodes_in_frontier: 34
max_depth: 31
cost_total: 31.0


Con 5 discos, la búsqueda voraz encontró el camino óptimo de 31 movimientos expandiendo muchos menos nodos que costo uniforme. Pero eso no está garantizado: la búsqueda voraz va siempre hacia lo que parece más cerca, y eso puede llevarla por un camino más largo. Lo vamos a ver con más discos al final de la notebook.

---

## Búsqueda A*

A* combina las dos ideas: el costo de llegar hasta el nodo, `g(n)`, y la estimación de lo que falta, `h(n)`:

`f(n) = g(n) + h(n)`

`f(n)` es una estimación del costo total del camino más barato que pasa por `n`. A* expande siempre el nodo de menor `f(n)`.

In [6]:
def a_star_search(number_disks: int = 5) -> tuple:
    """Búsqueda A*: la prioridad es f(n) = g(n) + h(n)."""
    return best_first_search(
        number_disks, lambda node: node.path_cost + misplaced_disks(node.state)
    )

In [7]:
solution, metrics = a_star_search(number_disks=5)

for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 169
states_visited: 169
nodes_in_frontier: 18
max_depth: 31
cost_total: 31.0


A* encontró el camino óptimo. Y en este caso no es suerte: está garantizado, porque nuestra heurística cumple dos propiedades.

**Es admisible**: nunca sobreestima lo que falta. Cada disco que está mal ubicado necesita, como mínimo, un movimiento para llegar a su lugar. Entonces, si hay `h(n)` discos mal ubicados, faltan al menos `h(n)` movimientos. Con una heurística admisible, A* encuentra siempre el camino más corto.

**Es consistente**: un movimiento mueve un solo disco, así que la cantidad de discos mal ubicados cambia, como mucho, en uno. Es decir, `h(n) ≤ costo(n, acción, n') + h(n')`. Con una heurística consistente, la primera vez que A* saca un estado de la frontera, lo hace por el camino más barato. Por eso podemos descartar los estados repetidos al sacarlos, como hacemos en `best_first_search()`.

¿Qué pasa si la heurística no aporta información? Si usamos `h(n) = 0` para todos los nodos, `f(n) = g(n)`, y A* es exactamente la búsqueda de costo uniforme:

In [8]:
_, metrics_zero = best_first_search(5, lambda node: node.path_cost)
_, metrics_a_star = a_star_search(number_disks=5)

print("A* con h(n) = 0, nodos expandidos:", metrics_zero["nodes_explored"])
print("A* con discos mal ubicados, nodos expandidos:", metrics_a_star["nodes_explored"])

A* con h(n) = 0, nodos expandidos: 216
A* con discos mal ubicados, nodos expandidos: 169


La heurística le ahorró a A*parte del trabajo: la información sobre el problema le permite descartar caminos que no prometen. Cuanto mejor estima una heurística admisible lo que falta, menos nodos expande A*.

---

## Comparando los algoritmos

Comparemos la búsqueda voraz y A* con la búsqueda de costo uniforme, para distintas cantidades de discos. El camino óptimo tiene 2ⁿ - 1 movimientos:

In [9]:
algorithms = {
    "Costo uniforme": lambda n: best_first_search(n, lambda node: node.path_cost),
    "Voraz": greedy_search,
    "A*": a_star_search,
}

print(f"{'Algoritmo':<16}{'Discos':>7}{'Óptimo':>8}{'Movimientos':>13}{'Nodos':>8}")
for number_disks in [5, 6, 7]:
    optimal = 2**number_disks - 1
    for name, algorithm in algorithms.items():
        _, metrics = algorithm(number_disks)

        moves = int(metrics["cost_total"])
        nodes = metrics["nodes_explored"]
        print(f"{name:<16}{number_disks:>7}{optimal:>8}{moves:>13}{nodes:>8}")
    print()

Algoritmo        Discos  Óptimo  Movimientos   Nodos
Costo uniforme        5      31           31     216
Voraz                 5      31           31     107
A*                    5      31           31     169

Costo uniforme        6      63           63     717
Voraz                 6      63           64     338
A*                    6      63           63     559

Costo uniforme        7     127          127    2091
Voraz                 7     127          131    1016


A*                    7     127          127    1809



Con 6 y 7 discos se ve la diferencia entre los dos algoritmos:

- La **búsqueda voraz** es la que menos nodos expande, pero encuentra caminos más largos que el óptimo. Va directo hacia lo que parece más cerca, sin mirar cuánto lleva recorrido.
- **A\*** siempre encuentra el camino óptimo, igual que costo uniforme, pero expandiendo menos nodos, gracias a la heurística.

Cuál conviene depende del problema: si necesitamos la mejor solución, A* con una heurística admisible. Si alcanza con una solución buena y rápida, la búsqueda voraz puede ser suficiente.

> 🧠 **Para pensar**: ¿se te ocurre una heurística mejor para la Torre de Hanoi? Recordá que, para que A* siga encontrando el camino óptimo, tiene que ser admisible. En el documento de [funciones heurísticas](../heuristic_functions.md) hay algunas ideas.